# Tunda Box Data Analyst Assessment

**Submission references:** [GitHub project](https://github.com/kimirifelix-droid/Data-Analyst-Assessment) · [Assessment responses](https://github.com/kimirifelix-droid/Data-Analyst-Assessment/blob/main/Assessment_Responses.md) · [Thought process](https://github.com/kimirifelix-droid/Data-Analyst-Assessment/blob/main/THOUGHT_PROCESS.md) · [README](https://github.com/kimirifelix-droid/Data-Analyst-Assessment/blob/main/README.md)

A self-contained Google Colab notebook using the assessment's invented sample data. Run cells from top to bottom. Assumptions: remove exact duplicate rows for order-level measures, normalize cities by trimming and lowercasing, count completed orders for revenue (NULL amounts do not contribute), and define repeat purchase as at least two distinct completed orders divided by all listed customers. The KES 185,000 amount is retained but flagged for verification.

In [ ]:
import math
import pandas as pd
from IPython.display import display

customers = pd.DataFrame([
    (101, 'Nairobi', '2026-01-14', 'Organic'),
    (102, 'nairobi', '2026-02-03', 'Referral'),
    (103, 'Mombasa', '2026-06-20', 'Paid Social'),
    (104, 'Kisumu', '2026-03-02', 'Organic'),
    (105, 'Nairobi ', '2026-03-15', 'Paid Social'),
    (106, 'Nakuru', '2026-04-01', 'Referral'),
    (107, 'Mombasa', '2026-04-18', 'Organic'),
    (108, 'Nairobi', '2026-05-09', 'Paid Social'),
    (109, 'Eldoret', '2026-06-21', 'Organic'),
    (110, 'Kisumu', '2026-07-05', 'Paid Social'),
], columns=['customer_id', 'city', 'signup_date', 'channel'])

orders = pd.DataFrame([
    (1001, 101, '2026-06-03', 'completed', 2400, 'M-Pesa'),
    (1002, 102, '2026-06-05', 'completed', 1850, 'M-Pesa'),
    (1003, 103, '2026-06-11', 'cancelled', 3200, 'Card'),
    (1004, 104, '2026-06-18', 'completed', 2100, 'M-Pesa'),
    (1005, 101, '2026-06-25', 'completed', 2650, 'Card'),
    (1006, 105, '2026-07-02', 'completed', 1500, 'M-Pesa'),
    (1006, 105, '2026-07-02', 'completed', 1500, 'M-Pesa'),
    (1007, 106, '2026-07-09', 'refunded', 2900, 'Card'),
    (1008, 102, '2026-07-14', 'completed', 3100, 'M-Pesa'),
    (1009, 107, '2026-07-21', 'completed', None, 'M-Pesa'),
    (1010, 101, '2026-07-28', 'completed', 2200, 'M-Pesa'),
    (1011, 108, '2026-08-04', 'completed', 1750, 'Card'),
    (1012, 103, '2026-08-09', 'completed', 2800, 'M-Pesa'),
    (1013, 104, '2026-08-15', 'completed', 185000, 'M-Pesa'),
    (1014, 105, '2026-08-19', 'cancelled', 1600, 'M-Pesa'),
    (1015, 102, '2026-08-27', 'completed', 2450, 'M-Pesa'),
    (1016, 106, '2026-09-03', 'completed', 3300, 'Card'),
    (1017, 101, '2026-09-08', 'completed', 2750, 'M-Pesa'),
    (1018, 108, '2026-09-14', 'refunded', 2000, 'M-Pesa'),
    (1019, 107, '2026-09-22', 'completed', 2350, 'Card'),
    (1020, 999, '2026-09-26', 'completed', 1900, 'M-Pesa'),
], columns=['order_id', 'customer_id', 'order_date', 'status', 'amount_kes', 'payment_method'])

orders['order_date'] = pd.to_datetime(orders['order_date'])
clean_orders = orders.drop_duplicates().copy()
customers['city_key'] = customers['city'].str.strip().str.lower()
print(f'Input rows: {len(orders)}; after removing exact duplicate rows: {len(clean_orders)}')
print(f'Exact duplicate rows removed: {len(orders) - len(clean_orders)}')

## S1 — Orders by status

SQL equivalent: select distinct full order records, group by `status`, and count. This avoids counting the exact duplicate order 1006 twice; conflicting records with the same ID would need investigation.

In [ ]:
status_counts = clean_orders.groupby('status').size().rename('order_count').sort_index()
display(status_counts.to_frame())
assert status_counts.to_dict() == {'cancelled': 2, 'completed': 16, 'refunded': 2}

## S2 — Completed revenue by normalized city

Join customers to deduplicated orders, retain completed statuses, and group by trimmed, case-insensitive city. SQL `SUM` and pandas sum both skip NULL amounts. The high Kisumu result is not necessarily a trustworthy business result until the KES 185,000 order is validated.

In [ ]:
completed = clean_orders.loc[clean_orders['status'].eq('completed')].copy()
city_revenue = (completed.merge(customers[['customer_id', 'city_key']], on='customer_id', how='inner')
                .groupby('city_key')['amount_kes'].sum(min_count=1)
                .sort_values(ascending=False)
                .rename('completed_revenue_kes'))
display(city_revenue.to_frame())
assert city_revenue.to_dict() == {'kisumu': 187100.0, 'nairobi': 20650.0, 'mombasa': 5150.0, 'nakuru': 3300.0}

## S3 — Customers who never ordered

“Ordered” means any order row, regardless of status. An anti-join / left join with no matching order identifies customers 109 and 110.

In [ ]:
ordered_customer_ids = set(clean_orders['customer_id'])
never_ordered = customers.loc[~customers['customer_id'].isin(ordered_customer_ids), ['customer_id', 'city']]
display(never_ordered)
assert never_ordered['customer_id'].tolist() == [109, 110]

## S4 — Repeat purchase rate

Definition: customers with at least two distinct completed orders divided by all customers in the customer table. The completed order with a missing amount still counts as an order.

In [ ]:
completed_counts = (completed.groupby('customer_id')['order_id'].nunique()
                    .reindex(customers['customer_id'], fill_value=0))
repeat_customers = completed_counts[completed_counts >= 2].index.tolist()
repeat_rate_pct = 100 * len(repeat_customers) / len(customers)
print(f'Repeat buyers: {repeat_customers}')
print(f'Repeat purchase rate: {len(repeat_customers)} / {len(customers)} = {repeat_rate_pct:.1f}%')
assert repeat_customers == [101, 102, 104, 107]
assert repeat_rate_pct == 40

## S5 — Monthly completed revenue and month-on-month change

Calendar-month revenue uses unique exact records. Percentage change is `(current - previous) / previous * 100`; the first month has no previous-month comparison.

In [ ]:
monthly = (completed.assign(month=completed['order_date'].dt.to_period('M').dt.to_timestamp())
           .groupby('month')['amount_kes'].sum(min_count=1)
           .rename('completed_revenue_kes').to_frame())
monthly['mom_change_pct'] = monthly['completed_revenue_kes'].pct_change() * 100
display(monthly.round({'mom_change_pct': 2}))
assert monthly['completed_revenue_kes'].tolist() == [9000.0, 6800.0, 192000.0, 10300.0]
print('August includes order 1013 (KES 185,000); validate before interpreting this spike.')

## D1 / D3 — Data quality and typical order value

Audit duplicate rows, missing completed amounts, unusually large amounts, unmatched customer IDs, and city formatting. Median is more robust than mean for a typical order in this sample.

In [ ]:
print('Missing completed amounts:')
display(completed.loc[completed['amount_kes'].isna(), ['order_id', 'customer_id', 'amount_kes']])
print('Amounts above KES 10,000 (review threshold for this small sample, not an automatic exclusion):')
display(completed.loc[completed['amount_kes'].gt(10000), ['order_id', 'customer_id', 'amount_kes']])
print('Order customer IDs without customer records:', sorted(set(clean_orders.customer_id) - set(customers.customer_id)))

valued_completed = completed['amount_kes'].dropna()
print(f'Non-null completed amounts: {len(valued_completed)}')
print(f'Median: KES {valued_completed.median():,.0f}')
print(f'Mean: KES {valued_completed.mean():,.0f}')
assert valued_completed.median() == 2400
assert valued_completed.mean() == 14540

## T2 — A/B test

Current: 50/1,000 = 5%; new: 60/1,000 = 6%. The absolute lift is 1 percentage point and the relative lift is 20%. The following is an approximate, two-sided pooled two-proportion z-test; interpret it alongside design quality, power, and practical impact.

In [ ]:
n_current, x_current = 1000, 50
n_new, x_new = 1000, 60
p_current, p_new = x_current / n_current, x_new / n_new
difference = p_new - p_current
pooled = (x_current + x_new) / (n_current + n_new)
se_null = math.sqrt(pooled * (1 - pooled) * (1 / n_current + 1 / n_new))
z_score = difference / se_null
p_value = math.erfc(abs(z_score) / math.sqrt(2))
se_ci = math.sqrt(p_current * (1 - p_current) / n_current + p_new * (1 - p_new) / n_new)
ci_low, ci_high = difference - 1.96 * se_ci, difference + 1.96 * se_ci
print(f'Current: {p_current:.1%}; new: {p_new:.1%}')
print(f'Absolute lift: {difference:.1%}; relative lift: {difference / p_current:.1%}')
print(f'z = {z_score:.3f}; two-sided p-value = {p_value:.3f}')
print(f'Approximate 95% CI for absolute lift: [{ci_low:.1%}, {ci_high:.1%}]')
assert round(p_value, 2) == 0.33

## Part 5 — Channel mix and overall conversion

Calculate each channel's conversion and session share, then compare the weighted total. Channel rates rise slightly; the lower-converting Paid Social channel contributes a much larger share of October sessions.

In [ ]:
traffic = pd.DataFrame([
    ('Organic', 40000, 2400, 38000, 2318),
    ('Paid Social', 10000, 200, 30000, 630),
], columns=['channel', 'sep_sessions', 'sep_orders', 'oct_sessions', 'oct_orders'])
traffic['sep_conversion_pct'] = 100 * traffic.sep_orders / traffic.sep_sessions
traffic['oct_conversion_pct'] = 100 * traffic.oct_orders / traffic.oct_sessions
traffic['sep_session_share_pct'] = 100 * traffic.sep_sessions / traffic.sep_sessions.sum()
traffic['oct_session_share_pct'] = 100 * traffic.oct_sessions / traffic.oct_sessions.sum()
display(traffic.round(2))
sep_total = traffic.sep_orders.sum() / traffic.sep_sessions.sum()
oct_total = traffic.oct_orders.sum() / traffic.oct_sessions.sum()
print(f'September overall: {traffic.sep_orders.sum():,} / {traffic.sep_sessions.sum():,} = {sep_total:.2%}')
print(f'October overall: {traffic.oct_orders.sum():,} / {traffic.oct_sessions.sum():,} = {oct_total:.2%}')
assert round(sep_total * 100, 1) == 5.2
assert round(oct_total * 100, 1) == 4.3

## Spreadsheet formulas

- Total completed M-Pesa revenue: `=SUMIFS(orders!E:E,orders!D:D,"completed",orders!F:F,"M-Pesa")`
- Customer city in `G2`: `=IFERROR(XLOOKUP(B2,customers!A:A,customers!B:B,"Unknown"),"Unknown")`
- XLOOKUP alternative: `=IFERROR(VLOOKUP(B2,customers!A:B,2,FALSE),"Unknown")`

Deduplicate order IDs first if the spreadsheet formula should reflect one record per order.